# Amazon Top-K Recommendation · C037–C076
## Ordering Artifact + B195 기반 추천 앙상블 · Kaggle Run All 전용

**이번 버전: 총 40개(C037~C076)**. 이전에 전달된 **C037~C059 노트북을 대체**합니다. 두 노트북을 동시에 실행하지 마세요.

### 이번에 반영한 추가 실험

| 실험 | 설명 | PDF/사진 Val Recall@20 |
|---|---|---:|
| B176 | LightGBM EXTRA 3-seed Average | 0.062098 |
| B187 | Temporal ItemKNN | 0.014149 |
| B188 | 기존 후보 + LightGBM Binary Top75 | 0.061850 |
| B189 | Temporal + LightGBM Binary Top50 | 0.062076 |
| B190 | Temporal + LightGBM Binary Top75 | 0.062244 |
| B191 | Temporal + LightGBM LambdaRank | 0.062003 |
| B192/B193/B194 | Ordering 단독 (이웃 5/20/50) | 0.012639 / 0.013543 / 0.013719 |
| **B195** | **Ordering + LightGBM Binary** | **0.071694 (Test 0.07819)** |

B195는 이미 Ordering 정보를 반영한 강한 모델입니다. 따라서 B195의 재학습 없이 기존 top100을 작은 비율로 섞는 **보수적인 블렌딩**과 **추가 Ordering 신호의 기여**를 분리 검증합니다. 성능 향상은 보장되지 않습니다.

### 필요한 Kaggle Input
1. **대회/검증 데이터 (7개)**: `train.csv`, `test.csv`, `train_reviews.jsonl`, `sample_submission.csv`, `val_train.csv`, `val_truth.csv`, `val_users.csv`.
2. **기존 top100 (39개)**: `B167, B170, B171, B176, B187, B188, B189, B190, B191, B192, B193, B194, B195`의 `val`, `inner`, `final` 파일. B187~B195는 `B195_val_top100.csv` 등의 이름을 사용. B167/B170/B171/B176은 원본 이름을 그대로 사용 가능하며 `.csv.gz` 압축도 지원합니다.
3. **선택 (정확한 Inner Ordering에 필수)**: 실제 Inner split의 관측 이력인 `inner_train.csv`. 없으면 Ordering을 Inner에 적용하지 않는 프록시 파일을 출력합니다.
4. `item_meta.jsonl`, CLIP 임베딩은 이번 코드에 **필수 아님** (추가 학습 없이 앙상블의 효과를 확인하는 실험이므로).

### GitHub 접근에 관한 제한
현재 실행 환경에서 해당 GitHub 저장소의 코드를 읽지 못했습니다. 이 코드는 **사용자가 공유한 PDF/사진 결과, 입력 파일 구조, 기존 독립 노트북 로직**을 기반으로 재구현한 것입니다. 원본 모델 학습 코드나 teamkit의 평가함수를 복제했다고 주장하지 않습니다.

### 누출 방지
- **Validation** Ordering 위치/이웃 후보 생성은 `train_reviews.jsonl`을 순서대로 읽되 **`val_train.csv`에 포함된 관측 `(user,item)` 쌍만 허용**합니다. `val_truth`는 Recall@20 계산에만 쓰고 학습, 점수, 가중치 결정에는 사용하지 않습니다.
- **Final** Ordering은 `train.csv`에 포함된 관측 이력만 사용합니다.
- **Inner**에서 `inner_train.csv`가 없으면 `inner`에 Ordering 신호를 적용하지 않고, 기존 예측의 랭크 블렌드만 수행합니다. Ordering 관련 실험의 Inner 결과는 **프록시**이므로 별도로 생성된 실제 Inner 예측처럼 사용하면 안 됩니다.
- 40개 실험 중 Val 상위 2개 선택은 공통 Validation에 대한 모델 선택이므로 그 점수가 별도 Test에서도 재현된다고 가정하지 않습니다.

**실행:** Kaggle `File → Import Notebook → Add Input datasets → Run All`. 다른 파일명을 쓴다면 바로 아래 `OVERRIDES`에 파일 경로를 지정하세요.
### 파일명 자동 감지 V3
`B187_temporal_val_top100.csv.gz`처럼 **B187~B195 ID가 파일명 중간에 있어도** 찾아요.
이름에 ID가 없으면 오연결하지 않고 실제 입력 경로를 출력하므로 `MODEL_FILE_PREFIXES` 또는 `OVERRIDES`에 지정해 주세요.


## 1. 설정 · 실험 명세 · 파일 입력

In [ ]:
# -*- coding: utf-8 -*-
"""Amazon C037-C076: post-B195 ablation, independent Kaggle run-all notebook source.
No validation-target fitting, no test labels. See notebook markdown for caveats.
"""
from __future__ import annotations
import ast, csv, json, math, random, re, gc
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass
import numpy as np
import pandas as pd

# ----------------- MODIFY ONLY IF KAGGLE INPUT PATHS ARE AMBIGUOUS -----------------
BASE_IDS = ('B167','B170','B171','B176','B187','B188','B189','B190','B191','B192','B193','B194','B195')
# Kaggle에 업로드된 CSV의 실제 접두어. 나머지 B187~B195는 ID 자체를 접두어로 사용.
MODEL_FILE_PREFIXES = {
    'B167':'D_FREEDOM_ENSEMBLE',
    'B170':'D_BLEND6',
    'B171':'TEAM_A_D_RRF',
    'B176':'TEAM_RERANK_EXTRA',
}
TOPK_EXPORT = 100
TOPK_METRIC = 20
AUTO_ROOTS = [Path('/kaggle/input'), Path('/kaggle/working'), Path('/mnt/data'), Path.cwd()]
OVERRIDES = {
    'train':None, 'test':None, 'val_train':None, 'val_truth':None, 'val_users':None,
    'reviews':None, 'sample_submission':None, 'inner_train':None,
    # Example: 'B195_val':'/kaggle/input/b195-predictions/B195_val_top100.csv'
}
OUT_DIR = Path('/kaggle/working/C037_C076_ablation') if Path('/kaggle/working').is_dir() else Path.cwd()/'C037_C076_ablation'

EXPERIMENTS = []
def add(expid, title, weights, group, method='inverse', k=20, adaptive=None, order=None, params=None):
    EXPERIMENTS.append(dict(id=expid, name=title, weights=weights, group=group,
                            method=method, k=k, adaptive=adaptive, order=order,
                            parameters=params or (f'method={method}; weights='+','.join(f'{a}:{b:g}' for a,b in weights.items())+
                                (f'; k={k}' if method=='rrf' else '')+
                                (f'; adaptive={adaptive}' if adaptive else ''))))
add('C037','B195 baseline',{'B195':1},'baseline',params='B195 only; existing Ordering+LightGBM Binary Top100')
add('C038','B190 baseline',{'B190':1},'baseline')
add('C039','B176 baseline',{'B176':1},'baseline')
add('C040','B194 Ordering only',{'B194':1},'baseline')
add('C041','B187 Temporal ItemKNN only',{'B187':1},'baseline')
add('C042','B195+B190 99/1',{'B195':.99,'B190':.01},'binary ensemble')
add('C043','B195+B190 97/3',{'B195':.97,'B190':.03},'binary ensemble')
add('C044','B195+B190 95/5',{'B195':.95,'B190':.05},'binary ensemble')
add('C045','B195+B190 90/10',{'B195':.90,'B190':.10},'binary ensemble')
add('C046','B195+B189 95/5',{'B195':.95,'B189':.05},'binary ensemble')
add('C047','B195+B191 95/5',{'B195':.95,'B191':.05},'binary ensemble')
add('C048','B195+B188 95/5',{'B195':.95,'B188':.05},'binary ensemble')
add('C049','B195+B176 97/3',{'B195':.97,'B176':.03},'legacy ensemble')
add('C050','B195+B171 97/3',{'B195':.97,'B171':.03},'legacy ensemble')
add('C051','B195+B170 97/3',{'B195':.97,'B170':.03},'legacy ensemble')
add('C052','B195+B167 97/3',{'B195':.97,'B167':.03},'legacy ensemble')
add('C053','B195+B194 98/2',{'B195':.98,'B194':.02},'ordering model ensemble')
add('C054','B195+B194 95/5',{'B195':.95,'B194':.05},'ordering model ensemble')
add('C055','B195+B193 98/2',{'B195':.98,'B193':.02},'ordering model ensemble')
add('C056','B195+B192 98/2',{'B195':.98,'B192':.02},'ordering model ensemble')
add('C057','B195+B187 98/2',{'B195':.98,'B187':.02},'temporal ensemble')
add('C058','B195+B190+B194 96/2/2',{'B195':.96,'B190':.02,'B194':.02},'multi-model')
add('C059','B195+B190+B176 94/4/2',{'B195':.94,'B190':.04,'B176':.02},'multi-model')
add('C060','B195+B189+B191 94/3/3',{'B195':.94,'B189':.03,'B191':.03},'multi-model')
add('C061','B195+B170+B167 95/3/2',{'B195':.95,'B170':.03,'B167':.02},'multi-model')
add('C062','B195+B190+B194+B176',{'B195':.93,'B190':.03,'B194':.02,'B176':.02},'multi-model')
add('C063','B195+B190 RRF20',{'B195':.95,'B190':.05},'RRF',method='rrf',k=20)
add('C064','B195+B190 RRF60',{'B195':.95,'B190':.05},'RRF',method='rrf',k=60)
add('C065','B195+B194 RRF20',{'B195':.95,'B194':.05},'RRF',method='rrf',k=20)
add('C066','B195+B190 score fusion',{'B195':.95,'B190':.05},'normalized raw score',method='score')
add('C067','B195+B194 score fusion',{'B195':.95,'B194':.05},'normalized raw score',method='score')
add('C068','B195+B190 adaptive overlap',{'B195':.93,'B190':.07},'adaptive',adaptive='overlap')
add('C069','B195+B194 adaptive activity',{'B195':.93,'B194':.07},'adaptive',adaptive='activity')
add('C070','B195+Ordering direct 2%',{'B195':.98,'ORDER20':.02},'ordering direct',params='first observed position; ±20 users; distance/pop penalty; rank weight .02')
add('C071','B195+Ordering direct 5%',{'B195':.95,'ORDER20':.05},'ordering direct',params='first observed position; ±20 users; distance/pop penalty; rank weight .05')
add('C072','B195+Ordering direct window50',{'B195':.95,'ORDER50':.05},'ordering direct',params='first observed position; ±50 users; distance/pop penalty; rank weight .05')
add('C073','B195+Ordering direct window5',{'B195':.95,'ORDER5':.05},'ordering direct',params='first observed position; ±5 users; distance/pop penalty; rank weight .05')
add('C074','B195+shuffled-order placebo',{'B195':.95,'ORDER_SHUFFLED':.05},'placebo',params='random seed42 shuffled user order; ±20; other settings C071')
add('C075','B195+B190+Ordering direct',{'B195':.94,'B190':.03,'ORDER20':.03},'ordering direct')
add('C076','B195+B190+B194+Ordering',{'B195':.91,'B190':.03,'B194':.03,'ORDER20':.03},'ordering direct')
assert len(EXPERIMENTS)==40 and EXPERIMENTS[0]['id']=='C037' and EXPERIMENTS[-1]['id']=='C076'

ALIASES = {
 'user': ('user_id','user','uid','reviewerid','reviewer_id','useridx','user_idx','reviewer','customer_id'),
 'item': ('item_id','item','iid','itemidx','item_idx','asin','parent_asin','parentasin','product_id','productid'),
 'rank': ('rank','position','pos','item_rank','ranking'),
 'score': ('score','pred_score','prediction_score','pred','weight','prob','probability','rating'),
 'items': ('items','item_ids','recommendations','recommendation','predictions','prediction','top100','top_100','top20','top_20','rec_items')
}
def normkey(x):return re.sub(r'[^a-z0-9]','',str(x).lower())
def pick_column(cols,group):
    cols={normkey(c):c for c in cols}
    for a in ALIASES[group]:
        if normkey(a) in cols:return cols[normkey(a)]
    return None

def clean_id(x):
    if x is None:return ''
    if isinstance(x,(float,np.floating)) and np.isnan(x):return ''
    z=str(x).strip()
    if z.lower() in ('','none','null','nan'):return ''
    if re.fullmatch(r'\d+\.0',z):return z[:-2]
    return z

def parse_items(x):
    if isinstance(x,(list,tuple,np.ndarray)):
        return [z for e in x if (z:=clean_id(e))]
    s=clean_id(x)
    if not s:return []
    if s.startswith('[') and s.endswith(']'):
        try:y=json.loads(s)
        except (ValueError,TypeError):
            try:y=ast.literal_eval(s)
            except (ValueError,TypeError,SyntaxError):y=None
        if isinstance(y,(list,tuple)):return [z for e in y if (z:=clean_id(e))]
    return [z for e in re.split(r'[,|;\s]+',s) if (z:=clean_id(e.strip('"\'')))]

def locate(name):
    if OVERRIDES.get(name):
        p=Path(OVERRIDES[name])
        if not p.is_file():raise FileNotFoundError(f'OVERRIDES[{name}] missing: {p}')
        return p
    names={'train':'train.csv','test':'test.csv','val_train':'val_train.csv',
           'val_truth':'val_truth.csv','val_users':'val_users.csv',
           'reviews':'train_reviews.jsonl','sample_submission':'sample_submission.csv',
           'inner_train':'inner_train.csv'}
    target=names.get(name,name)
    found=[]
    for root in AUTO_ROOTS:
        if root.is_dir():found+=list(root.rglob(target))
    found=[x for x in set(found) if x.is_file() and 'C037_C076_ablation' not in str(x)]
    if not found:return None
    if name.startswith('val_'):
        found.sort(key=lambda p:('validation' not in str(p).lower(),len(str(p)),str(p)))
    else:found.sort(key=lambda p:(int('validation' in str(p).lower()),len(str(p)),str(p)))
    if len(found)>1:print(f'[WARN] duplicate {name} -> using {found[0]} ; set OVERRIDES if incorrect')
    return found[0]

def _inventory_prediction_files():
    """Scan filenames once, supporting nested Kaggle datasets and compressed CSVs.

    Kaggle can store a prediction in e.g.
      /kaggle/input/B187_B195/preds/ordering_B195_v2_val_top100.csv.gz
    The *dataset folder* name is not used to decide model identity.
    """
    paths=[]
    for root in AUTO_ROOTS:
        root=Path(root)
        if not root.is_dir():
            continue
        for pat in ('*.csv','*.csv.gz','*.CSV','*.CSV.GZ'):
            paths.extend(root.rglob(pat))
    unique={str(p.resolve()):p for p in paths
            if p.is_file() and 'C037_C076_ablation' not in str(p)}
    return sorted(unique.values(),key=lambda p:(not str(p).startswith('/kaggle/input/'),len(str(p)),str(p)))

def _candidate_split(name,split):
    text=re.sub(r'\.csv(?:\.gz)?$','',name,flags=re.IGNORECASE)
    # Recognize val/inner/final delimited by underscore, space, period or hyphen.
    return re.search(r'(?<![a-z0-9])'+re.escape(split)+r'(?![a-z0-9])',text,re.I) is not None

def _candidate_model_id(name,eid):
    # Match model ID in *filename*, not Kaggle dataset directory, which may contain all IDs.
    return re.search(r'(?<![a-z0-9])'+re.escape(eid)+r'(?![0-9])',name,re.I) is not None

def locate_model(eid,split):
    """Find predictions when B187 etc. occur anywhere in the filename, not only as its prefix."""
    if split not in ('val','inner','final'):
        raise ValueError(f'Unknown split: {split}')
    key=f'{eid}_{split}'
    if OVERRIDES.get(key):
        path=Path(OVERRIDES[key])
        if not path.is_file():
            raise FileNotFoundError(f'OVERRIDES["{key}"] -> {path} not found')
        return path

    inventory=_inventory_prediction_files()
    prefixes=list(dict.fromkeys([eid,MODEL_FILE_PREFIXES.get(eid,eid)]))
    # First choose the exact traditional name, including aliases from B167 etc.
    for prefix in prefixes:
        for ext in ('.csv','.csv.gz'):
            expected=f'{prefix}_{split}_top100{ext}'
            exact=[p for p in inventory if p.name.lower()==expected.lower()]
            if exact:
                if len(exact)>1:
                    print(f'[WARN] duplicate {expected}: using {exact[0]}')
                return exact[0]

    # Then choose an unambiguous filename with this ID and split, e.g.
    # B187_temporal_..._val_top100.csv.gz, temporal_B187_val_top100.csv.gz.
    matches=[p for p in inventory if _candidate_model_id(p.name,eid)
             and _candidate_split(p.name,split)]
    if len(matches)>1:
        with_top=[p for p in matches if re.search(r'top[-_ ]?100',p.name,re.I)]
        if len(with_top)==1:
            matches=with_top
    if len(matches)==1:
        print(f'[AUTO-MAPPED] {key:<11} -> {matches[0].name}')
        return matches[0]
    elif len(matches)>1:
        raise FileNotFoundError(
            f'{key}: found MULTIPLE possible files; please specify OVERRIDES["{key}"]:\n'
            +'\n'.join('  '+str(p) for p in matches[:30]))

    # Keep explicit aliases available for files without ID in basename.
    # Do not guess an unnamed file's identity: B188/189/190 are too similar.
    sample=[p for p in inventory if _candidate_split(p.name,split)
            and re.search(r'top[-_ ]?100',p.name,re.I)]
    show='\n'.join('  '+str(p) for p in sample[:80])
    raise FileNotFoundError(
        f'{key}: no uniquely matching CSV found. Expected the model ID {eid} somewhere in '
        f'the filename (plus {split}), or exact prefix from MODEL_FILE_PREFIXES.\n'
        f'Files seen for split {split} ({len(sample)} shown, max 80):\n{show}\n'
        f'If real filenames do not contain {eid}, edit MODEL_FILE_PREFIXES["{eid}"] '
        f'or OVERRIDES["{key}"].')

def verify_model_inputs():
    """Print path-by-path mapping BEFORE running all 40 experiments."""
    records=[];missing=[]
    for eid in BASE_IDS:
        for split in ('val','inner','final'):
            try:
                p=locate_model(eid,split)
                records.append({'id':eid,'split':split,'filename':p.name,'path':str(p)})
            except FileNotFoundError as ex:
                missing.append(str(ex))
    for r in records:
        print(f"[FOUND] {r['id']:<4} {r['split']:<5} -> {r['filename']}")
    if missing:
        print('\n[DEBUG] List of all prediction-style files in Kaggle Input:')
        pred_files=[p for p in _inventory_prediction_files()
                    if re.search('top[-_ ]?100',p.name,re.I)]
        for p in pred_files[:100]:print(' ',p)
        raise FileNotFoundError(
            f'{len(missing)}/39 missing or ambiguous prediction inputs; first error:\n{missing[0]}\n'
            'Use the printed inventory to map unfamiliar filenames; no models were silently substituted.')
    selected=[r['path'] for r in records]
    duplicates=[x for x,c in Counter(selected).items() if c>1]
    if duplicates:
        raise ValueError(f'Same file incorrectly mapped to more than one model: {duplicates}')
    print(f'[OK] Verified {len(records)}/39 prediction files, including .csv.gz and filenames with suffixes.')
    return pd.DataFrame(records)

def read_interactions(path):
    hist=defaultdict(set)
    for chunk in pd.read_csv(path,dtype=str,keep_default_na=False,chunksize=250000):
        uc=pick_column(chunk.columns,'user');ic=pick_column(chunk.columns,'item')
        if uc is None or ic is None:raise ValueError(f'{path}: need user/item columns, found {list(chunk.columns)}')
        for u,i in zip(chunk[uc],chunk[ic]):
            u=clean_id(u);i=clean_id(i)
            if u and i:hist[u].add(i)
    print(f'[DATA] {path.name}: {len(hist):,} users; {sum(map(len,hist.values())):,} observed pairs')
    return hist

def read_users(path):
    d=pd.read_csv(path,dtype=str,keep_default_na=False)
    uc=pick_column(d.columns,'user') or d.columns[0]
    return list(dict.fromkeys(z for x in d[uc] if (z:=clean_id(x))))

def read_truth(path):
    truth=defaultdict(set)
    for chunk in pd.read_csv(path,dtype=str,keep_default_na=False,chunksize=250000):
        uc=pick_column(chunk.columns,'user') or chunk.columns[0]
        ic=pick_column(chunk.columns,'item')
        lc=pick_column(chunk.columns,'items')
        if ic:
            for u,i in zip(chunk[uc],chunk[ic]):
                u=clean_id(u);i=clean_id(i)
                if u and i:truth[u].add(i)
        else:
            if not lc:
                other=[x for x in chunk.columns if x!=uc]
                if len(other)!=1:raise ValueError(f'Unknown truth schema: {list(chunk.columns)}')
                lc=other[0]
            for u,items in zip(chunk[uc],chunk[lc]):
                u=clean_id(u)
                if u:truth[u].update(parse_items(items))
    return truth



In [ ]:
print('실험 개수:', len(EXPERIMENTS))
print('실험 범위:', EXPERIMENTS[0]['id'], '~', EXPERIMENTS[-1]['id'])
print('입력 모델:', ', '.join(BASE_IDS))
from IPython.display import display
display(pd.DataFrame(EXPERIMENTS)[['id', 'name', 'group', 'parameters']])
print('\n아래 39개 입력 파일명 사전 검증:')
verify_model_inputs()


## 2. top100 파일 파싱

지원 형식은 사용자-아이템 단위의 long CSV (`user_id,item_id,rank,score`), 사용자당 하나의 아이템 목록 (`user_id,items`), 사용자별 wide 열 (`item_1,item_2,...`)입니다. 숫자 `score`가 제공되지 않으면 score-fusion 실험은 랭크 기반 값으로 폴백합니다. 개인별 점수 정규화를 사용하므로 원시 점수 스케일이 모델마다 달라도 비교할 수 있습니다.

In [ ]:
@dataclass
class ModelPred:
    ranked: dict
    raw: dict
    score_users: int

# Accepted layouts: (user,item,rank,score) long; (user,items) list; (user,item1,item2..) wide.
def read_top100(path,maxlen=100):
    data=defaultdict(list); seq=0; fmt=None
    for chunk in pd.read_csv(path,dtype=str,keep_default_na=False,chunksize=200000):
        uc=pick_column(chunk.columns,'user') or chunk.columns[0]
        ic=pick_column(chunk.columns,'item');rc=pick_column(chunk.columns,'rank')
        sc=pick_column(chunk.columns,'score');lc=pick_column(chunk.columns,'items')
        if ic and ic!=uc:
            fmt='long'
            ranks=pd.to_numeric(chunk[rc],errors='coerce').to_numpy() if rc else None
            scores=pd.to_numeric(chunk[sc],errors='coerce').to_numpy() if sc else None
            for j,(u,i) in enumerate(zip(chunk[uc],chunk[ic])):
                u=clean_id(u);i=clean_id(i)
                if not u or not i:continue
                rank=float(ranks[j]) if ranks is not None and np.isfinite(ranks[j]) else 100000+seq
                score=float(scores[j]) if scores is not None and np.isfinite(scores[j]) else None
                data[u].append((rank,seq,i,score));seq+=1
        elif lc and lc!=uc:
            fmt='list'
            for u,items in zip(chunk[uc],chunk[lc]):
                u=clean_id(u)
                if not u:continue
                for j,i in enumerate(parse_items(items),1):data[u].append((j,seq,i,None));seq+=1
        else:
            fmt='wide';cols=[x for x in chunk.columns if x not in {uc,rc,sc}]
            if not cols:raise ValueError(f'Could not parse {path}, columns={list(chunk.columns)}')
            for record in chunk[[uc]+cols].itertuples(index=False,name=None):
                u=clean_id(record[0])
                if not u:continue
                for j,v in enumerate(record[1:],1):
                    i=clean_id(v)
                    if i:data[u].append((j,seq,i,None));seq+=1
    ranked={};raw={};score_users=0
    for u,rows in data.items():
        seen=set();items=[];scs=[]
        for r,n,i,sc in sorted(rows,key=lambda a:(a[0],a[1])):
            if i not in seen:
                seen.add(i);items.append(i);scs.append(sc)
                if len(items)>=maxlen:break
        ranked[u]=items
        if any(x is not None for x in scs):
            raw[u]=scs;score_users+=1
    print(f'[MODEL] {path.name}: {len(ranked):,} users; fmt={fmt}; numeric_score_users={score_users:,}')
    return ModelPred(ranked,raw,score_users)



## 3. Ordering Artifact와 Fusion

B192~B194가 이미 Ordering 단독 점수에서 개선을 보여준 점을 반영해, 순서 기반 후보를 B195의 일부 랭크에 추가하는 C070~C076을 설계했습니다. C074는 파일 사용자 순서를 seed42로 섞는 **placebo 대조군**입니다.

**B195가 Ordering을 이미 활용하고 있으므로 재활용 이득은 제한적일 수 있습니다.** 이 실험은 Ordering 효과를 보증하는 것이 아니라 상호 보완성이 남아 있는지 확인하는 것입니다.

Score fusion(C066·C067)은 사용 가능한 raw score를 유저별 minmax 정규화하고 rank 신호를 30% 섞습니다. Adaptive(C068·C069)는 각각 상위 20개 후보 중복률 및 사용자 관측 이력 수에 따라 소수 모델 가중치를 바꿉니다.

In [ ]:
@dataclass
class OrderState:
    by_first:list
    indices:dict
    scanned:int

# Validation histories and order positions contain ONLY pairs observed in val_train.
# Never use val_truth to produce candidates/order positions.
def make_order_states(reviews, observed):
    states={k:{} for k in observed}
    scanned={k:0 for k in observed}
    with open(reviews,'r',encoding='utf-8') as fh:
        uk=ik=None
        for lineno,line in enumerate(fh,1):
            if not line.strip():continue
            try:rec=json.loads(line)
            except json.JSONDecodeError:continue
            if not isinstance(rec,dict):continue
            if uk is None:
                uk=pick_column(rec.keys(),'user');ik=pick_column(rec.keys(),'item')
                if not uk or not ik:raise ValueError(f'{reviews} must contain user/item keys; found {list(rec)[:20]}')
                print('[ORDER] JSON keys:',uk,ik)
            u=clean_id(rec.get(uk));i=clean_id(rec.get(ik))
            if not u or not i:continue
            for split,hist in observed.items():
                if i not in hist.get(u,()):continue
                if u not in states[split]:states[split][u]=scanned[split]
                scanned[split]+=1
            if lineno%2000000==0:print(f'[ORDER] scanned {lineno:,} reviews')
    out={}
    for split,idx in states.items():
        order=sorted(idx,key=lambda u:(idx[u],u))
        out[split]=OrderState(order,{u:j for j,u in enumerate(order)},scanned[split])
        print(f'[ORDER] {split}: {len(order):,} users; {scanned[split]:,} observed rows matched')
    return out

def order_items(user,state,hist,pop,window=20,maxlen=100):
    pos=state.indices.get(user)
    if pos is None:return []
    ids=state.by_first; lo=max(0,pos-window);hi=min(len(ids),pos+window+1)
    score=defaultdict(float);seen=hist.get(user,set())
    for j in range(lo,hi):
        if j==pos:continue
        neighbor=hist.get(ids[j],())
        if not neighbor:continue
        d=abs(j-pos)
        strength=1/(d**.75 * math.sqrt(len(neighbor)))
        for item in neighbor:
            if item not in seen:
                score[item]+=strength / (1+math.log1p(pop.get(item,0)))
    return sorted(score,key=lambda i:(-score[i],i))[:maxlen]

def shuffled_state(state):
    order=list(state.by_first);random.Random(42).shuffle(order)
    return OrderState(order,{u:j for j,u in enumerate(order)},state.scanned)

def score_values(pred,user):
    items=pred.ranked.get(user,[])
    original=pred.raw.get(user)
    if not original or sum(x is not None and np.isfinite(x) for x in original)<2:
        return {i:2/(j+2) for j,i in enumerate(items)}
    finite=np.asarray([float(x) for x in original if x is not None and np.isfinite(x)],dtype=float)
    lo=float(finite.min());hi=float(finite.max());delta=hi-lo
    if delta<=1e-12:return {i:2/(j+2) for j,i in enumerate(items)}
    return {i: (0.70*((float(v)-lo)/delta)+0.30*(2/(j+2))) if v is not None and np.isfinite(v)
            else 0.30*(2/(j+2)) for j,(i,v) in enumerate(zip(items,original))}

def per_user_weights(exp,user,models,hist):
    w=exp['weights'].copy();a=exp.get('adaptive')
    if a=='overlap':
        r1=set(models['B195'].ranked.get(user,[])[:20]);r2=set(models['B190'].ranked.get(user,[])[:20]);
        ratio=len(r1&r2)/20
        # lower overlap -> less trust in B190, to avoid displacing B195's strong candidates.
        share=.07*(.3+.7*ratio)
        w={'B195':1-share,'B190':share}
    if a=='activity':
        n=len(hist.get(user,()))
        share=max(.015,min(.12,.07*math.sqrt(10/(n+1))))
        w={'B195':1-share,'B194':share}
    return w

def recommend(user,models,order_lists,exp,hist,fallback,maxlen=100):
    w=per_user_weights(exp,user,models,hist)
    den=sum(w.values())
    if den<=0:raise ValueError('invalid weights')
    fused=defaultdict(float);method=exp['method'];k=exp['k']
    b195=models['B195'].ranked.get(user,[])
    b195_ranks={i:j for j,i in enumerate(b195)}
    for model,weight in w.items():
        if weight<=0:continue
        if model.startswith('ORDER'):
            ranked=order_lists.get(model,[]);norm_scores=None
        else:
            ranked=models[model].ranked.get(user,[])
            norm_scores=score_values(models[model],user) if method=='score' else None
        for rank,item in enumerate(ranked[:maxlen],1):
            if method=='score' and norm_scores is not None:part=norm_scores.get(item,0)
            elif method=='rrf':part=(k+1)/(k+rank)
            else:part=2/(rank+1)
            fused[item]+=weight/den*part
    seen=hist.get(user,set())
    for item in seen:fused.pop(item,None)
    chosen=sorted(fused,key=lambda i:(-fused[i],b195_ranks.get(i,1000000),i))[:maxlen]
    if len(chosen)<maxlen:
        used=set(chosen)|seen
        for i in fallback:
            if i not in used:
                chosen.append(i);used.add(i)
            if len(chosen)>=maxlen:break
    if len(chosen)<20:raise ValueError(f'Only {len(chosen)} candidates user={user}. Check input catalog.')
    return chosen,fused

def sample_format(template,test_users,ranked,outpath):
    example=pd.read_csv(template,dtype=str,keep_default_na=False)
    uc=pick_column(example.columns,'user') or example.columns[0]
    more=[c for c in example.columns if c!=uc]
    if not more:raise ValueError('Sample submission has no item columns')
    if len(more)>1:
        K=len(more)
        if K>100:raise ValueError('Sample submission requires >100 items')
        rows=pd.DataFrame({uc:test_users})
        for j,c in enumerate(more):rows[c]=[ranked[u][j] for u in test_users]
    else:
        col=more[0];example_users=[clean_id(x) for x in example[uc]]
        vals=[str(x) for x in example[col] if str(x).strip()]
        val=vals[0] if vals else ''
        if len(example_users)>len(set(example_users)):
            counts=Counter(example_users)
            if len(set(counts.values()))!=1:raise ValueError('Template user repetition inconsistent')
            K=next(iter(counts.values()))
            rows=pd.DataFrame([(u,ranked[u][j]) for u in test_users for j in range(K)],columns=[uc,col])
        else:
            K=len(parse_items(val)) or 20
            if K>100:raise ValueError('Sample requires >100 items')
            if val.strip().startswith('['):fmt=lambda l:json.dumps(l,ensure_ascii=False)
            elif '|' in val:fmt=lambda l:'|'.join(l)
            elif ',' in val:fmt=lambda l:','.join(l)
            else:fmt=lambda l:' '.join(l)
            rows=pd.DataFrame({uc:test_users,col:[fmt(ranked[u][:K]) for u in test_users]})
    rows.to_csv(outpath,index=False)
    print(f'[SAVED] submission {outpath.name} rows={len(rows):,} cols={len(rows.columns)}')
    return rows



## 4. Run All · 3종 top100 및 상위 2개 제출 파일 생성

아래 셀에서 실험마다 `C037_val_top100.csv`, `C037_inner_top100.csv`, `C037_final_top100.csv`와 같은 형식으로 저장합니다. 모든 40개 실험을 실행해 **120개 Top100 CSV**가 나옵니다. 최상위 Val 2개는 `C###_submission.csv`도 생성합니다. 출력 경로: `/kaggle/working/C037_C076_ablation/`.

평가 공식은 **유저별 `hits@20 / len(val_truth[user])`의 평균**입니다. 이 값이 teamkit과 다를 수 있으므로 **C037의 B195 기준 재현값 0.071694를 반드시 확인**하세요. 수치가 크게 다르면 실제 평가 정의·파일 포맷을 먼저 교정해야 하며, 다른 실험의 성능 차이를 신뢰해서는 안 됩니다.

In [ ]:
def run_all():
    OUT_DIR.mkdir(exist_ok=True,parents=True)
    verify_model_inputs()
    paths={name:locate(name) for name in ('train','test','val_train','val_truth','val_users','reviews','sample_submission','inner_train')}
    for name,p in paths.items():print(f'[PATH] {name:<20} {p if p else "NOT FOUND"}')
    missing=[x for x in ('train','test','val_train','val_truth','val_users','reviews','sample_submission') if paths[x] is None]
    if missing:raise FileNotFoundError('Missing required input: '+', '.join(missing))
    inputs={split:read_interactions(paths[lookup]) for split,lookup in [('val','val_train'),('final','train'),('inner','inner_train')] if paths[lookup]}
    truth=read_truth(paths['val_truth'])
    val_users=read_users(paths['val_users']);test_users=read_users(paths['test'])
    if not val_users or not test_users:raise ValueError('Empty validation/test user list')
    missing_gt=[u for u in val_users if not truth.get(u)]
    if missing_gt:raise ValueError(f'No val ground truth for {len(missing_gt)} users; examples={missing_gt[:5]}')
    print(f'[DATA] val users={len(val_users):,}, test users={len(test_users):,}')
    order_states=make_order_states(paths['reviews'],inputs)
    if len(order_states['val'].by_first)<max(10,int(.2*len(val_users))):
        raise ValueError('Too few val users matched in train_reviews.jsonl. Check review user/item keys and matching IDs.')
    pops={split:Counter(i for seen in hist.values() for i in seen) for split,hist in inputs.items()}
    fallback={s:[i for i,_ in pop.most_common()] for s,pop in pops.items()}
    # If inner_train is missing, inner ordering is disabled and existing inner ranking is used.
    metrics={e['id']:{'sum':0.,'users':0,'hits':0,'hits_users':0} for e in EXPERIMENTS}
    best_ids=[];final_preds={}
    for split in ('val','inner','final'):
        print('\n'+'='*25+' '+split.upper()+' '+'='*25)
        models={model:read_top100(locate_model(model,split)) for model in BASE_IDS}
        users=val_users if split=='val' else test_users if split=='final' else list(models['B195'].ranked)
        missing_base=[u for u in users if u not in models['B195'].ranked]
        if missing_base:raise ValueError(f'{split}: B195 missing {len(missing_base)} user predictions, examples={missing_base[:4]}')
        hist=inputs.get(split,{})
        state=order_states.get(split)
        random_state=shuffled_state(state) if state else None
        pop=pops.get(split,Counter())
        fallback_split=fallback.get(split,fallback['val'])
        if split=='inner' and state is None:
            print('[IMPORTANT] inner_train.csv absent: ordering features disabled on inner, seen filtering disabled. '
                  'These inner files are RANK-ONLY PROXIES, NOT equivalent to a true inner split.')
        handles={};writers={}
        try:
            for exp in EXPERIMENTS:
                handle=(OUT_DIR/f"{exp['id']}_{split}_top100.csv").open('w',newline='',encoding='utf-8')
                handles[exp['id']]=handle;writers[exp['id']]=csv.writer(handle)
                writers[exp['id']].writerow(['user_id','item_id','rank','score'])
            for ui,u in enumerate(users,1):
                order_lists={}
                if state is not None:
                    order_lists['ORDER5']=order_items(u,state,hist,pop,window=5)
                    order_lists['ORDER20']=order_items(u,state,hist,pop,window=20)
                    order_lists['ORDER50']=order_items(u,state,hist,pop,window=50)
                    order_lists['ORDER_SHUFFLED']=order_items(u,random_state,hist,pop,window=20)
                for exp in EXPERIMENTS:
                    ranked,scores=recommend(u,models,order_lists,exp,hist,fallback_split)
                    wr=writers[exp['id']]
                    for j,i in enumerate(ranked,1):wr.writerow([u,i,j,f'{scores.get(i,0):.9g}'])
                    if split=='val':
                        target=truth[u]
                        hits=len(target.intersection(ranked[:TOPK_METRIC]))
                        m=metrics[exp['id']]
                        m['sum']+=hits/len(target);m['users']+=1;m['hits']+=hits;m['hits_users']+=bool(hits)
                    elif split=='final' and exp['id'] in best_ids:
                        final_preds.setdefault(exp['id'],{})[u]=ranked[:TOPK_METRIC]
                if ui%5000==0:print(f'[PROGRESS] {split}: {ui:,}/{len(users):,}')
        finally:
            for h in handles.values():h.close()
        print(f'[DONE] {split} top100 for {len(EXPERIMENTS)} experiments; users={len(users):,}')
        if split=='val':
            for exp in EXPERIMENTS:
                m=metrics[exp['id']]
                m['recall20']=m['sum']/m['users'] if m['users'] else float('nan')
            leaderboard=sorted(EXPERIMENTS,key=lambda e:(-metrics[e['id']]['recall20'],e['id']))
            best_ids=[e['id'] for e in leaderboard[:2]]
            base=metrics['C037']['recall20']
            print('[VAL] C037(B195) recall@20:',round(base,6),'  reference PDF 0.071694')
            if abs(base-.071694)>.001:
                print('*** WARNING *** B195 reference score mismatch. Validate input versions, column order, user IDs, '
                      'heldout exclusion, and teamkit Recall@20 before interpreting other results.')
            print('[VAL] Top ten:')
            for e in leaderboard[:10]:print(' ',e['id'],f"{metrics[e['id']]['recall20']:.6f}",e['name'])
        del models;gc.collect()
    base=metrics['C037']['recall20'];rows=[]
    for e in EXPERIMENTS:
        m=metrics[e['id']]
        rows.append({'experiment_id':e['id'],'model_or_combination':e['name'],'parameters':e['parameters'],
                     'val_recall@20':round(m['recall20'],8),'test':'',
                     'delta_vs_B195':round(m['recall20']-base,8),'group':e['group'],
                     'val_hit_users':m['hits_users'],'val_user_count':m['users']})
    ablation=pd.DataFrame(rows)
    ablation.to_csv(OUT_DIR/'C037_C076_ablation.csv',index=False,encoding='utf-8-sig')
    ablation.sort_values('val_recall@20',ascending=False).to_csv(OUT_DIR/'C037_C076_leaderboard.csv',index=False,encoding='utf-8-sig')
    best_ids_for_sub=[]
    # Deduplicate only if the two selected experiments yield exactly the same Top20 for all users.
    for exp in sorted(EXPERIMENTS,key=lambda e:(-metrics[e['id']]['recall20'],e['id'])):
        eid=exp['id']
        if eid not in final_preds:continue
        if any(final_preds[eid]==final_preds[other] for other in best_ids_for_sub):
            print(f'[NOTICE] Selected {eid} final predictions duplicate another selected model; both retained as requested')
        best_ids_for_sub.append(eid)
        if len(best_ids_for_sub)==2:break
    for eid in best_ids_for_sub:
        sample_format(paths['sample_submission'],test_users,final_preds[eid],OUT_DIR/f'{eid}_submission.csv')
    (OUT_DIR/'EXPERIMENT_NOTES.txt').write_text(
      'B195 baseline reference 0.071694. Validation is used ONLY for model selection, never to fit weights or reranker.\n'
      'INNER PROXY WARNING: '+('inner_train unavailable, ordering/seen filtering disabled on inner.\n' if 'inner' not in inputs else 'inner_train supplied; ordering built only on inner observed pairs.\n')+
      'Input models: '+', '.join(BASE_IDS)+'\nTop2 selected: '+', '.join(best_ids_for_sub)+'\n',encoding='utf-8')
    print('\n'+'='*50+'\nABLATION TABLE\n'+ablation[['experiment_id','model_or_combination','parameters','val_recall@20','test']].to_string(index=False))
    print('\nTOP2',best_ids_for_sub,'  output:',OUT_DIR)
    return ablation,best_ids_for_sub


In [ ]:
ablation, best_two = run_all()

## 5. PDF와 동일한 Ablation 표 · 검증 및 결과 확인

In [ ]:
from IPython.display import display
ref = 0.071694
base = float(ablation.set_index('experiment_id').loc['C037','val_recall@20'])
print(f'B195 reported={ref:.6f} / C037 reproduced={base:.6f} / diff={base-ref:+.6f}')
print('TOP 2:', ', '.join(best_two))
print('Output directory:',OUT_DIR)
print('Created top100 CSV:',len(list(OUT_DIR.glob('C???_*_top100.csv'))))
print('Submission CSV:',[f'{x}_submission.csv' for x in best_two])
pdf_columns=['experiment_id','model_or_combination','parameters','val_recall@20','test']
display(ablation[pdf_columns].style.format({'val_recall@20':'{:.6f}'}))
print('Top 15 + B195 delta:')
display(ablation.sort_values('val_recall@20',ascending=False).head(15)[['experiment_id','model_or_combination','val_recall@20','delta_vs_B195']])
assert len(ablation)==40
assert len(list(OUT_DIR.glob('C???_*_top100.csv')))==120
assert all((OUT_DIR/f'{eid}_submission.csv').is_file() for eid in best_two)
print('최종 검증 완료 (파일 개수, 파일 존재 여부)')

## 6. 일괄 ZIP 다운로드

`summary_and_submissions.zip`은 최종 PDF 스타일 표와 제출 파일 두 개만 담습니다. `all_top100.zip`은 큰 파일이므로 필요할 때 `CREATE_ALL_TOP100_ZIP=True`로 설정한 후 아래 셀을 다시 실행하세요. 디스크 절약을 위해 기본값은 False입니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
summary = OUT_DIR/'C037_C076_summary_and_submissions.zip'
with ZipFile(summary,'w',ZIP_DEFLATED) as zf:
    for name in ['C037_C076_ablation.csv','C037_C076_leaderboard.csv','EXPERIMENT_NOTES.txt']:
        zf.write(OUT_DIR/name,arcname=name)
    for eid in best_two:
        zf.write(OUT_DIR/f'{eid}_submission.csv',arcname=f'{eid}_submission.csv')
print('Summary ZIP:',summary)
CREATE_ALL_TOP100_ZIP=False
if CREATE_ALL_TOP100_ZIP:
    top_zip=OUT_DIR/'C037_C076_all_top100.zip'
    with ZipFile(top_zip,'w',ZIP_DEFLATED) as zf:
        for p in sorted(OUT_DIR.glob('C???_*_top100.csv')):zf.write(p,arcname=p.name)
    print('All Top100 ZIP:',top_zip)

### 체크 포인트

- `C037`이 **0.071694**와 충분히 일치하는지 확인합니다. 다르면 teamkit 함수 또는 `B195_val_top100.csv` 포맷/버전을 우선 확인하세요.
- `C070~C073`과 `C074`를 비교해 순서 정보가 단순 후보 증가를 넘어 실제로 유효한지 봅니다.
- Validation 성능이 같거나 아주 비슷한 실험은 무작정 더 좋은 것으로 판단하지 말고 사용자별 개선 분포를 확인하세요.
- Inner 기록을 팀원에게 전달하기 전에 `EXPERIMENT_NOTES.txt`에 `INNER PROXY WARNING`이 있는지 확인하세요.
- 이 노트북은 새 재랭커 학습 실험이 아니라 기존 예측을 활용한 **앙상블·순서 활용 Ablation**입니다.